# Experimento MLflow: clasificación de vinos con artefactos

Este notebook entrena un modelo de clasificación sobre el dataset `wine`, registra métricas, parámetros, tags y artefactos en MLflow.

In [ ]:
!pip install "mlflow-skinny==3.0.0" scikit-learn pandas matplotlib joblib

In [ ]:
import json
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from matplotlib import pyplot as plt

In [ ]:
# Conexión a MLflow
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("clasificacion-vinos-avanzada")

In [ ]:
# Carga de datos
data = load_wine()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

In [ ]:
# Pipeline del modelo
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=42,
        class_weight="balanced"
    ))
])

In [ ]:
with mlflow.start_run(run_name="rf-wine-v2") as run:
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.25)
    mlflow.log_param("class_weight", "balanced")

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision_weighted", precision)
    mlflow.log_metric("recall_weighted", recall)
    mlflow.log_metric("f1_weighted", f1)

    mlflow.set_tag("dataset", "wine")
    mlflow.set_tag("framework", "scikit-learn")
    mlflow.set_tag("task", "classification")
    mlflow.set_tag("owner", "bootcamp")

    # Resumen del dataset
    dataset_summary = {
        "n_samples": int(len(X)),
        "n_features": int(X.shape[1]),
        "features": list(feature_names),
        "target_names": list(target_names),
        "train_size": int(len(X_train)),
        "test_size": int(len(X_test))
    }

    with open("dataset_summary.json", "w") as f:
        json.dump(dataset_summary, f, indent=2)
    mlflow.log_artifact("dataset_summary.json", artifact_path="data")

    # Matriz de confusión
    with open("confusion_matrix.json", "w") as f:
        json.dump(cm.tolist(), f)
    mlflow.log_artifact("confusion_matrix.json", artifact_path="metrics")

    # Gráfico
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, cmap="Blues")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
    plt.xticks(range(len(target_names)), target_names)
    plt.yticks(range(len(target_names)), target_names)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.title("Matriz de confusión - vino")
    plt.tight_layout()
    plt.savefig("confusion_matrix_plot.png")
    plt.close()
    mlflow.log_artifact("confusion_matrix_plot.png", artifact_path="plots")

    mlflow.sklearn.log_model(
        sk_model=pipe,
        artifact_path="model",
        input_example=X_test[:5],
        registered_model_name="wine-classifier-rf"
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 weighted: {f1:.4f}")

## Qué deberías ver en MLflow

- Parámetros del modelo
- Métricas: accuracy, precision, recall, f1
- Tags del experimento
- Artefactos: modelo, JSON con la confusión, JSON del dataset y gráfica
- Registro del modelo como `wine-classifier-rf`